# 딥러닝 기초 1~4강 통합 실습

이 Notebook은 강의에서 배운 핵심 연산을 작은 입력으로 다시 구현하고, 바로 옆의 공개 검사로 확인하도록 구성했습니다. 함수의 입력 검사, 반복되는 반환 형식, 비교용 fixture처럼 개념 학습의 중심이 아닌 코드는 가능한 한 준비되어 있습니다.

위에서 아래로 진행하세요. 각 실습에서는 설명과 작은 예를 먼저 읽고, 구현 셀의 `TODO`만 채운 뒤 예제와 `check_e##()`을 실행합니다. 검사가 실패하면 traceback의 첫 실패부터 확인하고, 마지막 정리 칸에는 값과 shape가 왜 그렇게 나왔는지 적으세요.

필요할 때만 [학습 기록](../../til/2026/08/2026-08-24.md)과 [강의 목차](../../materials/private/kant-deep-learning-basics/INDEX.md)를 복습하면 됩니다. 이 Notebook만으로도 모든 공개 검사의 조건을 알 수 있습니다.


In [15]:
# 공통 준비
from typing import Any

import numpy as np
import torch
from torch import nn


## 실습 1. 문제에 맞는 시작점과 출력 형식 정하기

규칙이나 단순 기준선을 먼저 둘지 판단하고, 데이터 layout과 task에 맞는 출력 형식을 정합니다. 이 실습은 이미 작성한 학습자 코드를 그대로 보존합니다.

### 준비되어 있는 부분

예제 입력과 경계값·오류 입력을 포함한 검사가 준비되어 있습니다.

### 직접 완성할 부분

이미 완성한 `recommend_start`, `model_family_for_layout`, `task_contract` 구현을 그대로 사용합니다.

### 요구사항

- `stable_rule=True`이면 다른 조건보다 먼저 `rule_first`을 반환하고, 그렇지 않으면서 `raw_unstructured=True`이고 `labeled_count >= 5_000`이면 `deep_learning_candidate`, 나머지는 `simple_baseline_first`를 반환해야 합니다.
- layout은 `image → cnn`, `sequence → transformer`, `tabular → mlp`로 연결해야 합니다.
- `image`, `sequence`, `tabular`가 아닌 layout은 `ValueError`로 거부해야 합니다.
- `task_contract`가 반환하는 dict의 key는 정확히 `output_shape`, `target_shape`, `target_dtype`, `loss`여야 합니다.
- regression은 output `(B,1)`, target `(B,1)` `torch.float32`, `MSELoss`를 사용해야 합니다.
- binary는 output `(B,1)`, target `(B,1)` `torch.float32`, `BCEWithLogitsLoss`를 사용해야 합니다.
- multiclass는 output `(B,C)`, target `(B,)` `torch.long`, `CrossEntropyLoss`를 사용해야 합니다.
- `batch_size`는 양수여야 하고 multiclass의 `num_classes`는 2 이상이어야 하며 지원하지 않는 task와 잘못된 값은 `ValueError`로 거부해야 합니다.

### 작은 예

`4_999`와 `5_000`은 서로 다른 분기로 들어갑니다. 먼저 어느 문자열이 나와야 하는지 예상해 보세요.

<details><summary>힌트 1</summary>

분기 순서를 위에서 아래로 읽고 가장 강한 조건인 안정적인 규칙을 먼저 확인합니다.

</details>

<details><summary>힌트 2</summary>

task별 정보는 같은 네 key를 가진 dict로 만들면 차이를 비교하기 쉽습니다.

</details>


In [2]:
# TODO: 아래 함수들을 직접 구현하세요
def recommend_start(stable_rule: bool, raw_unstructured: bool, labeled_count: int) -> str:
    """Return rule_first, deep_learning_candidate, or simple_baseline_first."""
    if stable_rule:
        return "rule_first"

    if raw_unstructured and labeled_count >= 5000:
        return "deep_learning_candidate"

    return "simple_baseline_first"


def model_family_for_layout(layout: str) -> str:
    """Map image, sequence, or tabular to its usual baseline model family."""
    if layout == "image":
        return "cnn"

    if layout == "sequence":
        return "transformer"

    if layout == "tabular":
        return "mlp"

    raise ValueError("image, tabular, sequence 중 하나가 입력으로 들어와야 합니다.")


def task_contract(task: str, batch_size: int, num_classes: int | None = None) -> dict[str, Any]:
    """Return output_shape, target_shape, target_dtype, and loss for a task."""
    if batch_size <= 0:
        raise ValueError("batch_size는 1 이상이여야 한다.")

    if task == "regression":
        output_shape = (batch_size, 1)
        target_shape = (batch_size, 1)
        target_dtype = torch.float32
        loss = "MSELoss"
    elif task == "binary":
        output_shape = (batch_size, 1)
        target_shape = (batch_size, 1)
        target_dtype = torch.float32
        loss = "BCEWithLogitsLoss"
    elif task == "multiclass":
        if not num_classes or num_classes < 2:
            raise ValueError("multiclass의 num_classes인자는 2이상이여야 한다.")

        output_shape = (batch_size, num_classes)
        target_shape = (batch_size,)
        target_dtype = torch.long
        loss = "CrossEntropyLoss"
    else:
        raise ValueError("지원하지 않는 task입니다.")

    return {
        "output_shape": output_shape,
        "target_shape": target_shape,
        "target_dtype": target_dtype,
        "loss": loss
    }


In [3]:
# 예제 입력으로 동작을 살펴보세요
scenarios = [
    (True, True, 28_000),
    (False, True, 28_000),
    (False, True, 120),
]
print("starts:", [recommend_start(*case) for case in scenarios])
print("image family:", model_family_for_layout("image"))
print("sequence family:", model_family_for_layout("sequence"))
print("multiclass:", task_contract("multiclass", batch_size=5, num_classes=3))


starts: ['rule_first', 'deep_learning_candidate', 'simple_baseline_first']
image family: cnn
sequence family: transformer
multiclass: {'output_shape': (5, 3), 'target_shape': (5,), 'target_dtype': torch.int64, 'loss': 'CrossEntropyLoss'}


In [4]:
# 구현을 마친 뒤 이 셀을 실행하세요
def check_e01() -> None:
    # 기본 동작
    np.testing.assert_equal(recommend_start(True, True, 28_000), "rule_first")
    np.testing.assert_equal(recommend_start(False, True, 28_000), "deep_learning_candidate")
    np.testing.assert_equal(recommend_start(False, True, 120), "simple_baseline_first")
    np.testing.assert_equal(model_family_for_layout("image"), "cnn")
    np.testing.assert_equal(model_family_for_layout("sequence"), "transformer")
    np.testing.assert_equal(model_family_for_layout("tabular"), "mlp")
    multiclass = task_contract("multiclass", 5, 3)
    np.testing.assert_equal(sorted(multiclass), ["loss", "output_shape", "target_dtype", "target_shape"])
    np.testing.assert_equal(multiclass["output_shape"], (5, 3))
    np.testing.assert_equal(multiclass["target_shape"], (5,))
    np.testing.assert_equal(multiclass["target_dtype"], torch.long)
    np.testing.assert_equal(multiclass["loss"], "CrossEntropyLoss")

    # 경계값
    np.testing.assert_equal(recommend_start(False, True, 4_999), "simple_baseline_first")
    np.testing.assert_equal(recommend_start(False, True, 5_000), "deep_learning_candidate")
    binary = task_contract("binary", 1)
    np.testing.assert_equal(binary["output_shape"], (1, 1))
    np.testing.assert_equal(binary["target_shape"], (1, 1))
    np.testing.assert_equal(binary["target_dtype"], torch.float32)
    np.testing.assert_equal(binary["loss"], "BCEWithLogitsLoss")
    regression = task_contract("regression", 2)
    np.testing.assert_equal(regression["output_shape"], (2, 1))
    np.testing.assert_equal(regression["target_shape"], (2, 1))
    np.testing.assert_equal(regression["target_dtype"], torch.float32)
    np.testing.assert_equal(regression["loss"], "MSELoss")

    # 잘못된 입력
    try:
        model_family_for_layout("audio")
    except ValueError:
        pass
    else:
        raise AssertionError("unknown layouts must fail")
    try:
        task_contract("binary", 0)
    except ValueError:
        pass
    else:
        raise AssertionError("non-positive batch_size must fail")
    try:
        task_contract("multiclass", 2)
    except ValueError:
        pass
    else:
        raise AssertionError("multiclass num_classes below 2 must fail")
    try:
        task_contract("ranking", 2)
    except ValueError:
        pass
    else:
        raise AssertionError("unknown tasks must fail")


check_e01()


### 확인 결과 정리

선택 복습입니다. 원한다면 왜 `5_000`을 보편적인 딥러닝 기준이 아니라 이 실습 안의 판단 기준으로만 읽어야 하는지 메모해도 좋습니다. 이 메모는 실습 완료 조건이 아닙니다.


## 실습 2. Tensor 정보를 한눈에 기록하기

Tensor를 계산에 넣기 전에 shape, 차원 수, dtype, device를 작은 카드로 확인합니다.

### 준비되어 있는 부분

함수 시그니처, Tensor 입력 검사, 반환 dict의 key와 입력 이름 전달은 준비되어 있습니다.

### 직접 완성할 부분

Tensor에서 `shape`, `ndim`, `dtype`, `device`를 읽는 네 표현만 채웁니다.

### 요구사항

- 반환된 `name`은 함수에 입력한 이름과 같아야 합니다.
- 반환된 `shape`는 일반 tuple, `ndim`은 차원 수, `dtype`과 `device`는 각각 Tensor 속성의 문자열이어야 합니다.
- `tensor`가 PyTorch Tensor가 아니면 `TypeError`로 거부해야 합니다.
- `(B,C)` logits를 쓰는 `CrossEntropyLoss` 오류를 진단할 때 target card가 `(B,1)`이거나 `torch.float32`라면 target을 `(B,)` `torch.long` class index로 고쳐야 한다고 설명해야 합니다.

### 작은 예

`torch.zeros(2, 3)`이라면 shape는 `(2, 3)`, ndim은 `2`입니다. dtype과 device가 어떤 문자열이 될지도 예상하세요.

<details><summary>힌트 1</summary>

Tensor가 이미 가진 네 속성을 하나씩 확인하세요.

</details>

<details><summary>힌트 2</summary>

shape는 `tuple(...)`, dtype과 device는 `str(...)`로 직렬화합니다.

</details>


In [16]:
def tensor_card(name: str, tensor: torch.Tensor) -> dict[str, object]:
    """Return a compact card for one Tensor."""
    if not isinstance(tensor, torch.Tensor):
        raise TypeError("tensor must be a PyTorch Tensor")

    # TODO: Tensor의 네 속성을 읽어 아래 변수에 저장하세요.
    shape = tuple(tensor.shape)
    ndim = tensor.ndim
    dtype = str(tensor.dtype)
    device = str(tensor.device)

    return {
        "name": name,
        "shape": shape,
        "ndim": ndim,
        "dtype": dtype,
        "device": device,
    }


In [17]:
# 서로 rank가 다른 Tensor 카드 예시
for name, tensor in [
    ("score", torch.tensor(1.0)),
    ("ids", torch.tensor([1, 2, 3])),
    ("batch", torch.zeros(2, 3)),
]:
    print(tensor_card(name, tensor))


{'name': 'score', 'shape': (), 'ndim': 0, 'dtype': 'torch.float32', 'device': 'cpu'}
{'name': 'ids', 'shape': (3,), 'ndim': 1, 'dtype': 'torch.int64', 'device': 'cpu'}
{'name': 'batch', 'shape': (2, 3), 'ndim': 2, 'dtype': 'torch.float32', 'device': 'cpu'}


In [18]:
# 구현을 마친 뒤 이 셀을 실행하세요
def check_e02() -> None:
    card = tensor_card("pred", torch.zeros(3, 4, dtype=torch.float32))
    np.testing.assert_equal(card["name"], "pred")
    np.testing.assert_equal(card["shape"], (3, 4))
    np.testing.assert_equal(card["ndim"], 2)
    np.testing.assert_equal(card["dtype"], "torch.float32")
    np.testing.assert_equal(card["device"], "cpu")

    scalar = tensor_card("score", torch.tensor(1.0))
    np.testing.assert_equal(scalar["shape"], ())

    try:
        tensor_card("bad", np.zeros((2, 3)))
    except TypeError:
        pass
    else:
        raise AssertionError("PyTorch Tensor가 아니면 TypeError여야 합니다")


check_e02()


### 확인 결과 정리

다중 분류 target 오류를 Tensor 카드로 진단해 보세요. logits가 `(8,3)`인데 target card가 `shape=(8,1)`, `dtype=torch.float32`라면 무엇을 어떻게 고쳐야 하나요?

**작성:** 아직 작성하지 않음


## 실습 3. 단일 이미지를 batch 형태로 바꾸기

샘플 하나에도 batch 축을 앞에 두어 모델이 항상 같은 입력 규칙을 받게 합니다.

### 준비되어 있는 부분

허용 rank 검사와 반환 함수의 형태는 준비되어 있습니다.

### 직접 완성할 부분

단일 이미지에는 첫 축을 추가하고, 이미 batch인 입력은 그대로 두는 분기를 구현합니다. 검사 뒤에는 세 대표 layout의 축과 자주 쓰는 모델 계열을 연결합니다.

### 요구사항

- rank 3 입력 `(C,H,W)`에는 0번 위치에 batch 축을 하나 추가하고, rank 4 입력 `(B,C,H,W)`는 값과 shape를 그대로 유지해야 합니다.
- 입력 rank가 3이나 4가 아니면 `ValueError`로 거부해야 합니다.
- `B`는 모든 layout에서 batch 수이고, `(B,C,H,W)`의 나머지 축은 channel·높이·너비, `(B,L,D)`는 sequence 길이·embedding 차원, `(B,F)`는 feature 수로 읽으며 각각 CNN·Transformer·MLP가 자주 쓰이는 대표 연결이라고 설명해야 합니다.

### 작은 예

`(1, 2, 3)` 이미지 한 장은 `(1, 1, 2, 3)`이 되고 `(4, 1, 2, 3)` batch는 그대로입니다.

<details><summary>힌트 1</summary>

입력의 `ndim`으로 단일 샘플과 batch를 구분하세요.

</details>

<details><summary>힌트 2</summary>

새 축은 샘플의 기존 축들보다 앞인 0번 위치에 추가합니다.

</details>


In [19]:
def ensure_image_batch(inputs: torch.Tensor) -> torch.Tensor:
    """Normalize one image or an image batch to `(B,C,H,W)`."""
    if inputs.ndim not in (3, 4):
        raise ValueError("inputs must have rank 3 or 4")

    # TODO: 단일 이미지에만 batch 축을 추가하세요.
    normalized = inputs.unsqueeze(0) if inputs.ndim == 3 else inputs
    return normalized


In [20]:
# 단일 이미지와 이미 만들어진 batch
single_image = torch.arange(6, dtype=torch.float32).reshape(1, 2, 3)
image_batch = torch.zeros(4, 1, 2, 3)
print("single ->", ensure_image_batch(single_image).shape)
print("batch  ->", ensure_image_batch(image_batch).shape)


single -> torch.Size([1, 1, 2, 3])
batch  -> torch.Size([4, 1, 2, 3])


In [21]:
# 구현을 마친 뒤 이 셀을 실행하세요
def check_e03() -> None:
    single = torch.arange(6, dtype=torch.float32).reshape(1, 2, 3)
    normalized = ensure_image_batch(single)
    np.testing.assert_equal(tuple(normalized.shape), (1, 1, 2, 3))
    torch.testing.assert_close(normalized[0], single)

    batch = torch.randn(4, 1, 2, 3)
    kept = ensure_image_batch(batch)
    np.testing.assert_equal(tuple(kept.shape), (4, 1, 2, 3))
    torch.testing.assert_close(kept, batch)

    try:
        ensure_image_batch(torch.zeros(2, 3))
    except ValueError:
        pass
    else:
        raise AssertionError("rank 2 입력은 ValueError여야 합니다")


check_e03()


### 확인 결과 정리

세 대표 layout의 축 의미를 모델 계열과 연결해 보세요. `(B,C,H,W)`, `(B,L,D)`, `(B,F)`에서 각 문자가 뜻하는 것과 CNN·Transformer·MLP가 어디에 자주 쓰이는지 적으세요. 이 연결은 절대적인 규칙이 아니라 대표적인 선택임도 덧붙이세요.

**작성:** 아직 작성하지 않음


## 실습 4. 회귀 target의 위험한 broadcasting 막기

`(B,1)` prediction과 `(B,)` target이 조용히 `(B,B)`로 확장되는 버그를 계산 전에 막습니다.

### 준비되어 있는 부분

prediction rank와 최종 shape가 맞는지 검사하고 residual을 계산하는 코드는 준비되어 있습니다.

### 직접 완성할 부분

target이 `(B,)`일 때 마지막 singleton 축을 추가하는 한 연산만 구현합니다.

### 요구사항

- prediction이 `(B,1)`이고 target이 `(B,)`이면 target의 1번 위치에 singleton 축을 추가해 `(B,1)`로 만든 뒤 sample별 residual `prediction - target`을 계산해야 합니다.
- prediction은 rank 2이면서 마지막 길이가 1이어야 하고 정규화 뒤 target shape가 prediction과 다르면 `ValueError`로 거부해야 합니다.

### 작은 예

`pred.shape == (4,1)`이고 `target.shape == (4,)`이면 바로 빼지 말고 target을 `(4,1)`로 바꿉니다.

<details><summary>힌트 1</summary>

batch 축 0은 그대로 두고 그 뒤에 길이 1인 축을 만드세요.

</details>

<details><summary>힌트 2</summary>

`unsqueeze`에 전달할 차원 번호를 shape 그림으로 먼저 정하세요.

</details>


In [ ]:
def regression_residual(prediction: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Return sample-wise residuals without accidental `(B,B)` broadcasting."""
    if prediction.ndim != 2 or prediction.shape[1] != 1:
        raise ValueError("prediction must have shape (B,1)")

    if target.ndim == 1:
        # TODO: target의 batch 축 뒤에 singleton 축을 추가하세요.
        aligned_target = NotImplemented
    else:
        aligned_target = target

    if aligned_target.shape != prediction.shape:
        raise ValueError("target must align exactly with prediction")
    return prediction - aligned_target


In [ ]:
# 잘못 빼면 (3,3), 올바르게 맞추면 (3,1)
pred = torch.tensor([[0.2], [0.7], [1.1]])
target = torch.tensor([0.0, 1.0, 1.0])
print("unsafe:", (pred - target).shape)
print("safe  :", regression_residual(pred, target).shape)


In [ ]:
# 구현을 마친 뒤 이 셀을 실행하세요
def check_e04() -> None:
    pred = torch.tensor([[0.2], [0.7], [1.1]])
    target = torch.tensor([0.0, 1.0, 1.0])
    residual = regression_residual(pred, target)
    np.testing.assert_equal(tuple(residual.shape), (3, 1))
    torch.testing.assert_close(residual, torch.tensor([[0.2], [-0.3], [0.1]]))

    already_aligned = regression_residual(pred, target.unsqueeze(1))
    torch.testing.assert_close(already_aligned, residual)

    try:
        regression_residual(torch.zeros(3, 2), target)
    except ValueError:
        pass
    else:
        raise AssertionError("prediction의 마지막 길이가 1이 아니면 ValueError여야 합니다")

    try:
        regression_residual(pred, torch.zeros(2, 1))
    except ValueError:
        pass
    else:
        raise AssertionError("batch 크기가 다르면 ValueError여야 합니다")


check_e04()


### 확인 결과 정리

선택 복습입니다. 원한다면 왜 `(B,)`와 `(B,1)`이 원소 수는 같아도 뺄셈에서 다른 축 의미를 만드는지 메모해도 좋습니다. 이 메모는 실습 완료 조건이 아닙니다.


## 실습 5. model·input·target의 device 맞추기

모델이 있는 장치를 기준으로 입력과 target을 함께 이동해 device mismatch를 예방합니다.

### 준비되어 있는 부분

모델의 첫 parameter에서 device를 찾고, 이동된 Tensor를 dict로 포장하는 코드는 준비되어 있습니다.

### 직접 완성할 부분

입력과 target 각각에 `.to(device)`를 적용하고 반환값을 새 변수에 저장합니다.

### 요구사항

- 입력과 target은 각각 `.to(device)`의 반환값을 저장해 모델 parameter와 같은 device로 이동해야 하며 값과 shape는 유지되어야 합니다.

### 작은 예

CPU 모델이면 두 Tensor도 CPU에 남고, CUDA 모델이면 둘 다 같은 CUDA 장치로 이동해야 합니다.

<details><summary>힌트 1</summary>

`device` 변수는 이미 준비되어 있으므로 두 Tensor에 같은 메서드를 호출하세요.

</details>

<details><summary>힌트 2</summary>

`.to(...)`는 이동된 Tensor를 반환하므로 호출 결과를 반드시 저장합니다.

</details>


In [ ]:
def move_batch_to_model(
    model: nn.Module,
    inputs: torch.Tensor,
    target: torch.Tensor,
) -> dict[str, object]:
    """Move a batch to the device of the model's first parameter."""
    device = next(model.parameters()).device

    # TODO: 입력과 target을 model device로 이동하세요.
    moved_inputs = NotImplemented
    moved_target = NotImplemented

    return {
        "inputs": moved_inputs,
        "target": moved_target,
        "device": str(device),
    }


In [ ]:
# 현재 환경에서 동작하는 model device를 기준으로 확인합니다.
device_model = nn.Linear(3, 2)
device_batch = move_batch_to_model(
    device_model,
    torch.arange(6, dtype=torch.float32).reshape(2, 3),
    torch.tensor([0, 1], dtype=torch.long),
)
print(device_batch["device"], device_batch["inputs"].device, device_batch["target"].device)


In [ ]:
# 구현을 마친 뒤 이 셀을 실행하세요
def check_e05() -> None:
    model = nn.Linear(3, 2)
    inputs = torch.arange(6, dtype=torch.float32).reshape(2, 3)
    target = torch.tensor([0, 1], dtype=torch.long)
    moved = move_batch_to_model(model, inputs, target)
    expected_device = next(model.parameters()).device
    np.testing.assert_equal(moved["device"], str(expected_device))
    np.testing.assert_equal(str(moved["inputs"].device), str(expected_device))
    np.testing.assert_equal(str(moved["target"].device), str(expected_device))
    torch.testing.assert_close(moved["inputs"].cpu(), inputs)
    torch.testing.assert_close(moved["target"].cpu(), target)


check_e05()


### 확인 결과 정리

선택 복습입니다. 원한다면 Tensor와 모델의 device가 모두 같아야 하는 이유를 실제 연산의 피연산자 관점에서 메모해도 좋습니다. 이 메모는 실습 완료 조건이 아닙니다.


## 실습 6. Linear 계산과 퍼셉트론 예측 구현하기

저장된 weight의 방향을 읽어 `nn.Linear`와 같은 계산을 만들고, 퍼셉트론의 raw score에서 class를 정합니다.

### 준비되어 있는 부분

rank·feature 수 검증과 반환 dict 조립은 준비되어 있습니다.

### 직접 완성할 부분

`x @ weight.T + bias` 계산과 `X @ w + b` 뒤 0 기준 prediction을 구현합니다.

### 요구사항

- `manual_linear`는 입력과 `nn.Linear`의 저장 weight·bias로 `inputs @ weight.T + bias`를 계산해 모듈 호출과 같은 `(B,out_features)` 값을 반환해야 합니다.
- `perceptron_output`은 `inputs @ weights + bias`로 `(B,)` raw logits를 만들고 `logits >= 0`인 위치를 1로 바꾼 `torch.long` prediction을 반환해야 합니다.
- 입력 rank와 feature 수가 각 계산의 weight와 맞지 않으면 `ValueError`로 거부해야 합니다.

### 작은 예

입력 `(4,3)`과 `Linear(3,2)`의 weight `(2,3)`을 곱하려면 weight를 전치하며 출력은 `(4,2)`입니다.

<details><summary>힌트 1</summary>

행렬 곱의 안쪽 차원이 맞도록 저장된 weight의 shape를 먼저 쓰세요.

</details>

<details><summary>힌트 2</summary>

퍼셉트론의 logit은 확률이 아니며 0 이상인지 비교해 0/1 정수 Tensor를 만듭니다.

</details>


In [ ]:
def manual_linear(inputs: torch.Tensor, layer: nn.Linear) -> torch.Tensor:
    """Reproduce one `nn.Linear` call from its stored parameters."""
    if inputs.ndim != 2 or inputs.shape[-1] != layer.in_features:
        raise ValueError("inputs must be rank 2 with the layer's in_features")

    # TODO: nn.Linear와 같은 행렬 계산을 작성하세요.
    output = NotImplemented
    return output


def perceptron_output(
    inputs: torch.Tensor,
    weights: torch.Tensor,
    bias: float | torch.Tensor,
) -> dict[str, torch.Tensor]:
    """Return raw logits and zero-threshold predictions."""
    if inputs.ndim != 2 or weights.ndim != 1 or inputs.shape[-1] != weights.shape[0]:
        raise ValueError("inputs and weights have incompatible shapes")

    # TODO: raw logits와 0 기준 prediction을 계산하세요.
    logits = NotImplemented
    prediction = NotImplemented
    return {"logits": logits, "prediction": prediction}


In [ ]:
# 재현 가능한 작은 Linear와 퍼셉트론 입력
torch.manual_seed(6)
linear = nn.Linear(3, 2)
linear_inputs = torch.randn(4, 3)
perceptron_inputs = torch.tensor([[1.0, 2.0], [-1.0, 0.0], [2.0, -1.0]])
print("manual Linear shape:", manual_linear(linear_inputs, linear).shape)
print(perceptron_output(perceptron_inputs, torch.tensor([1.0, -0.5]), 0.0))


In [ ]:
# 구현을 마친 뒤 이 셀을 실행하세요
def check_e06() -> None:
    torch.manual_seed(6)
    layer = nn.Linear(3, 2)
    inputs = torch.randn(4, 3)
    manual = manual_linear(inputs, layer)
    torch.testing.assert_close(manual, layer(inputs))
    np.testing.assert_equal(tuple(layer.weight.shape), (2, 3))
    np.testing.assert_equal(tuple(layer.bias.shape), (2,))
    np.testing.assert_equal(tuple(manual.shape), (4, 2))

    values = torch.tensor([[1.0, 2.0], [-1.0, 0.0], [2.0, -1.0]])
    result = perceptron_output(values, torch.tensor([1.0, -0.5]), 0.0)
    torch.testing.assert_close(result["logits"], torch.tensor([0.0, -1.0, 2.5]))
    torch.testing.assert_close(result["prediction"], torch.tensor([1, 0, 1]))
    np.testing.assert_equal(result["prediction"].dtype, torch.long)

    try:
        manual_linear(torch.zeros(2, 4), layer)
    except ValueError:
        pass
    else:
        raise AssertionError("in_features가 다르면 ValueError여야 합니다")


check_e06()


### 확인 결과 정리

선택 복습입니다. 원한다면 `Linear(3,2)`의 weight가 `(2,3)`인데도 입력에 곱할 때 전치가 필요한 이유를 메모해도 좋습니다. 이 메모는 실습 완료 조건이 아닙니다.


## 실습 7. 이미지 MLP의 계층과 forward 연결하기

이미지 batch의 첫 축을 보존하며 flatten, 은닉층, ReLU, 출력층을 순서대로 연결합니다.

### 준비되어 있는 부분

클래스 시그니처와 `super().__init__()` 호출은 준비되어 있습니다.

### 직접 완성할 부분

생성자에 네 모듈을 등록하고, forward에서 이전 단계의 출력을 다음 단계로 전달합니다.

### 요구사항

- 생성자에서 `self.flatten`, `self.hidden`, `self.relu`, `self.output`이라는 속성에 각각 `nn.Flatten(start_dim=1)`, `nn.Linear(input_dim, hidden_dim)`, `nn.ReLU()`, `nn.Linear(hidden_dim, num_classes)`를 등록해야 합니다.
- forward는 `flatten → hidden Linear → ReLU → output Linear` 순서로 직전 결과를 전달하고 `(B,C,H,W)` 입력의 batch 축을 유지한 `(B,num_classes)` raw logits를 반환해야 합니다.

### 작은 예

입력 `(2,1,2,3)`은 flatten 뒤 `(2,6)`, hidden 뒤 `(2,4)`, 출력층 뒤 `(2,3)`이 됩니다.

<details><summary>힌트 1</summary>

`nn.Flatten(start_dim=1)`은 batch 축만 남깁니다.

</details>

<details><summary>힌트 2</summary>

forward에서는 항상 직전 줄에서 만든 Tensor를 다음 모듈에 넣으세요.

</details>


In [ ]:
class ImageMLP(nn.Module):
    def __init__(self, image_shape: tuple[int, int, int], hidden_dim: int, num_classes: int):
        super().__init__()
        input_dim = image_shape[0] * image_shape[1] * image_shape[2]

        # TODO: flatten, hidden, ReLU, output 모듈을 등록하세요.
        self.flatten = NotImplemented
        self.hidden = NotImplemented
        self.relu = NotImplemented
        self.output = NotImplemented

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        # TODO: 네 모듈을 올바른 순서로 연결하세요.
        raise NotImplementedError("forward 순서를 구현하세요")


In [ ]:
# 작은 1x2x3 이미지를 쓰면 중간 feature 수를 눈으로 추적하기 쉽습니다.
torch.manual_seed(7)
images = torch.randn(2, 1, 2, 3)
image_mlp = ImageMLP((1, 2, 3), hidden_dim=4, num_classes=3)
print("images:", images.shape)
print("logits:", image_mlp(images).shape)


In [ ]:
# 구현을 마친 뒤 이 셀을 실행하세요
def check_e07() -> None:
    torch.manual_seed(7)
    model = ImageMLP((1, 2, 3), hidden_dim=4, num_classes=3)
    np.testing.assert_equal(isinstance(model.flatten, nn.Flatten), True)
    np.testing.assert_equal(model.flatten.start_dim, 1)
    np.testing.assert_equal((model.hidden.in_features, model.hidden.out_features), (6, 4))
    np.testing.assert_equal(isinstance(model.relu, nn.ReLU), True)
    np.testing.assert_equal((model.output.in_features, model.output.out_features), (4, 3))

    logits = model(torch.randn(2, 1, 2, 3))
    np.testing.assert_equal(tuple(logits.shape), (2, 3))
    one_logit = model(torch.randn(1, 1, 2, 3))
    np.testing.assert_equal(tuple(one_logit.shape), (1, 3))


check_e07()


### 확인 결과 정리

선택 복습입니다. 원한다면 왜 `forward`에서 원본 `images`가 아니라 직전 중간 표현을 다음 층에 넘겨야 하는지 메모해도 좋습니다. 이 메모는 실습 완료 조건이 아닙니다.


## 실습 8. 연속 affine 층을 하나로 합성하기

활성화 함수가 없는 두 Linear가 결국 하나의 affine 변환과 같음을 수치로 확인합니다.

### 준비되어 있는 부분

두 층의 parameter shape 검증과 fixture가 준비되어 있습니다.

### 직접 완성할 부분

합성 weight와 합성 bias의 두 식을 도출해 채웁니다.

### 요구사항

- 첫 층 `x @ W1.T + b1`과 둘째 층 `h @ W2.T + b2`는 `W_new = W2 @ W1`, `b_new = b1 @ W2.T + b2`로 합성해 같은 출력을 만들어야 합니다.

### 작은 예

첫 층이 `x @ W1.T + b1`, 둘째가 `h @ W2.T + b2`라면 `h` 식을 둘째 식에 대입해 보세요.

<details><summary>힌트 1</summary>

입력에 곱해지는 행렬끼리 먼저 묶으면 저장 weight의 합성 순서를 알 수 있습니다.

</details>

<details><summary>힌트 2</summary>

첫 bias도 둘째 weight를 통과한 뒤 마지막 bias와 더해집니다.

</details>


In [ ]:
def compose_affine(first: nn.Linear, second: nn.Linear) -> tuple[torch.Tensor, torch.Tensor]:
    """Return parameters of the affine map equivalent to `second(first(x))`."""
    if first.out_features != second.in_features:
        raise ValueError("the two Linear layers are not composable")

    # TODO: 두 affine 층의 합성 weight와 bias를 계산하세요.
    combined_weight = NotImplemented
    combined_bias = NotImplemented
    return combined_weight, combined_bias


In [ ]:
# 두 층과 합성 층의 출력을 비교할 작은 입력
torch.manual_seed(8)
first_affine = nn.Linear(3, 4)
second_affine = nn.Linear(4, 2)
affine_inputs = torch.randn(5, 3)
combined_weight, combined_bias = compose_affine(first_affine, second_affine)
print(combined_weight.shape, combined_bias.shape)


In [ ]:
# 구현을 마친 뒤 이 셀을 실행하세요
def check_e08() -> None:
    torch.manual_seed(8)
    first = nn.Linear(3, 4)
    second = nn.Linear(4, 2)
    inputs = torch.randn(5, 3)
    weight, bias = compose_affine(first, second)
    expected = second(first(inputs))
    actual = inputs @ weight.T + bias
    torch.testing.assert_close(actual, expected)
    np.testing.assert_equal(tuple(weight.shape), (2, 3))
    np.testing.assert_equal(tuple(bias.shape), (2,))


check_e08()


### 확인 결과 정리

선택 복습입니다. 원한다면 중간에 ReLU가 들어가면 두 층을 같은 방식으로 하나의 affine 식으로 합칠 수 없는 이유를 메모해도 좋습니다. 이 메모는 실습 완료 조건이 아닙니다.


## 실습 9. ReLU 특징으로 XOR 표현하기

두 입력의 차이를 양방향으로 ReLU에 통과시켜 직선 하나로 분리할 수 없던 XOR을 표현합니다.

### 준비되어 있는 부분

입력 검사, feature를 prediction으로 바꾸는 함수, ReLU 활성 비율 진단 함수는 준비되어 있습니다.

### 직접 완성할 부분

`ReLU(x1-x2)`와 `ReLU(x2-x1)` 두 feature를 batch의 두 번째 축에 쌓고, 검사 결과의 활성 비율과 gradient 신호로 dead ReLU 위험을 해석합니다.

### 요구사항

- `xor_features`는 각 sample에서 `ReLU(x1-x2)`와 `ReLU(x2-x1)`를 계산해 `(B,2)`로 쌓고, 두 feature 합이 양수이면 XOR prediction 1이 되게 해야 합니다.
- `relu_diagnostics`는 ReLU가 입력 shape를 유지하는지, 각 feature에서 pre-activation이 양수인 sample 비율, 음수·0 위치는 0이고 양수 위치는 1인 gradient 신호를 반환해야 합니다.
- 어떤 feature의 `positive_ratio`가 계속 0이고 `gradient_signal`도 모두 0이면 그 ReLU 뉴런은 음수·0 구간에서 학습 신호를 받지 못하는 dead ReLU 상태일 수 있다고 해석해야 합니다.

### 작은 예

`(0,1)`에서는 첫 차이는 0, 둘째 차이는 1이므로 두 feature의 합이 양수입니다. `(1,1)`에서는 둘 다 0입니다.

<details><summary>힌트 1</summary>

두 열을 `inputs[:, 0]`과 `inputs[:, 1]`로 분리하세요.

</details>

<details><summary>힌트 2</summary>

두 결과를 sample별 한 행으로 만들려면 새 feature 축에 쌓아야 합니다.

</details>


In [ ]:
def xor_features(inputs: torch.Tensor) -> torch.Tensor:
    """Build two ReLU features for a `(B,2)` XOR input."""
    if inputs.ndim != 2 or inputs.shape[1] != 2:
        raise ValueError("inputs must have shape (B,2)")

    # TODO: XOR을 드러내는 두 ReLU feature를 만드세요.
    features = NotImplemented
    return features


def xor_predict(inputs: torch.Tensor) -> dict[str, torch.Tensor]:
    """Provided wrapper that turns the hidden features into labels."""
    features = xor_features(inputs)
    prediction = (features.sum(dim=1) > 0).long()
    return {"features": features, "prediction": prediction}


def relu_diagnostics(pre_activation: torch.Tensor) -> dict[str, torch.Tensor]:
    """Provided helper for values, shape, and per-feature active ratio."""
    activated = torch.relu(pre_activation)
    positive_ratio = (pre_activation > 0).float().mean(dim=0)
    gradient_signal = (pre_activation > 0).to(pre_activation.dtype)
    return {
        "activated": activated,
        "positive_ratio": positive_ratio,
        "gradient_signal": gradient_signal,
    }


In [ ]:
# 네 XOR 점과 작은 pre-activation batch
xor_inputs = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
print(xor_predict(xor_inputs))
print(relu_diagnostics(torch.tensor([[-1.0, 2.0], [3.0, -4.0]])))


In [ ]:
# 구현을 마친 뒤 이 셀을 실행하세요
def check_e09() -> None:
    inputs = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
    result = xor_predict(inputs)
    torch.testing.assert_close(
        result["features"],
        torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 0.0]]),
    )
    torch.testing.assert_close(result["prediction"], torch.tensor([0, 1, 1, 0]))

    pre = torch.tensor([[-1.0, 2.0], [3.0, -4.0]])
    diagnostic = relu_diagnostics(pre)
    np.testing.assert_equal(tuple(diagnostic["activated"].shape), (2, 2))
    torch.testing.assert_close(diagnostic["activated"], torch.tensor([[0.0, 2.0], [3.0, 0.0]]))
    torch.testing.assert_close(diagnostic["positive_ratio"], torch.tensor([0.5, 0.5]))
    torch.testing.assert_close(diagnostic["gradient_signal"], torch.tensor([[0.0, 1.0], [1.0, 0.0]]))


check_e09()


### 확인 결과 정리

활성 비율과 gradient 신호로 dead ReLU를 진단해 보세요. 한 feature의 `positive_ratio`가 여러 batch에서 계속 0이고 `gradient_signal`도 모두 0이라면 parameter 학습에 어떤 일이 생기는지 적으세요.

**작성:** 아직 작성하지 않음


## 실습 10. 이진·다중 레이블 출력 해석하기

raw logits를 loss에 직접 넣는 학습 경로와 Sigmoid·threshold를 쓰는 추론 경로를 분리합니다.

### 준비되어 있는 부분

binary·multilabel task와 target shape·dtype, threshold 범위를 검사하고 결과 dict를 만드는 코드는 준비되어 있습니다.

### 직접 완성할 부분

Sigmoid probability, `>= threshold` prediction, raw-logits BCE loss의 세 표현을 채웁니다.

### 요구사항

- probability는 raw logits에 원소별 Sigmoid를 적용해 logits와 같은 shape로 만들어야 합니다.
- prediction은 `probability >= threshold`인 원소를 1로 정하고 logits와 같은 shape의 `torch.long` Tensor로 만들어야 합니다.
- loss는 probability가 아니라 raw logits와 같은 shape의 `torch.float32` target을 `BCEWithLogitsLoss`에 직접 넣어 계산해야 합니다.
- binary는 logits와 target이 `(B,1)`, multilabel은 `(B,C)`여야 하고 target은 `torch.float32`, threshold는 0 이상 1 이하여야 하며 위반하면 `ValueError`로 거부해야 합니다.

### 작은 예

logit 0의 Sigmoid는 0.5이므로 threshold가 0.5이면 `>=` 규칙에 따라 prediction은 1입니다.

<details><summary>힌트 1</summary>

확률은 `torch.sigmoid`로 만들지만 loss에는 그 확률 변수를 넣지 않습니다.

</details>

<details><summary>힌트 2</summary>

boolean 비교 결과를 `torch.long`으로 바꾸면 0/1 prediction이 됩니다.

</details>


In [ ]:
def _validate_bce_inputs(
    task: str,
    logits: torch.Tensor,
    target: torch.Tensor,
    threshold: float,
) -> None:
    if task not in {"binary", "multilabel"}:
        raise ValueError("task must be binary or multilabel")
    if logits.ndim != 2 or target.shape != logits.shape:
        raise ValueError("logits and target must have the same rank-2 shape")
    if task == "binary" and logits.shape[1] != 1:
        raise ValueError("binary logits must have shape (B,1)")
    if target.dtype != torch.float32:
        raise ValueError("BCE target must use torch.float32")
    if not 0.0 <= threshold <= 1.0:
        raise ValueError("threshold must be in [0,1]")


def binary_outputs(
    task: str,
    logits: torch.Tensor,
    target: torch.Tensor,
    threshold: float = 0.5,
) -> dict[str, object]:
    """Return loss, probability, and prediction for BCE tasks."""
    _validate_bce_inputs(task, logits, target, threshold)

    # TODO: raw logits에서 Sigmoid probability를 계산하세요.
    probability = NotImplemented

    # TODO: threshold를 포함하는 0/1 prediction을 만드세요.
    prediction = NotImplemented

    # TODO: raw logits로 BCE loss를 계산하세요.
    loss = NotImplemented
    return {"loss": loss, "probability": probability, "prediction": prediction}


In [ ]:
# 0 logit이 threshold 경계 0.5에 놓이는 작은 예
binary_logits = torch.tensor([[-2.0], [0.0], [2.0]])
binary_target = torch.tensor([[0.0], [1.0], [1.0]], dtype=torch.float32)
print(binary_outputs("binary", binary_logits, binary_target, threshold=0.5))


In [ ]:
# 구현을 마친 뒤 이 셀을 실행하세요
def check_e10() -> None:
    logits = torch.tensor([[-2.0], [0.0], [2.0]])
    target = torch.tensor([[0.0], [1.0], [1.0]], dtype=torch.float32)
    result = binary_outputs("binary", logits, target, threshold=0.5)
    torch.testing.assert_close(result["probability"], torch.sigmoid(logits))
    torch.testing.assert_close(result["prediction"], torch.tensor([[0], [1], [1]]))
    np.testing.assert_equal(result["prediction"].dtype, torch.long)
    torch.testing.assert_close(result["loss"], nn.BCEWithLogitsLoss()(logits, target))

    multilabel_logits = torch.tensor([[0.0, -1.0, 2.0]])
    multilabel_target = torch.tensor([[1.0, 0.0, 1.0]])
    multilabel = binary_outputs("multilabel", multilabel_logits, multilabel_target)
    np.testing.assert_equal(tuple(multilabel["prediction"].shape), (1, 3))

    try:
        binary_outputs("binary", logits, target.long())
    except ValueError:
        pass
    else:
        raise AssertionError("BCE target이 long이면 ValueError여야 합니다")

    try:
        binary_outputs("binary", logits, target, threshold=1.1)
    except ValueError:
        pass
    else:
        raise AssertionError("threshold 범위가 잘못되면 ValueError여야 합니다")


check_e10()


### 확인 결과 정리

선택 복습입니다. 원한다면 왜 학습 loss 경로에서는 Sigmoid를 먼저 적용하지 않고 추론 경로에서만 probability를 만드는지 메모해도 좋습니다. 이 메모는 실습 완료 조건이 아닙니다.


## 실습 11. 다중 분류 출력 해석하기

class별 raw logits에서 sample별 분포와 예측 class를 만들되 CrossEntropyLoss의 입력 경로는 분리합니다.

### 준비되어 있는 부분

logits·target shape, dtype, class 범위를 검사하고 결과를 dict로 포장하는 코드는 준비되어 있습니다.

### 직접 완성할 부분

마지막 class 축 Softmax, 같은 축 argmax, raw-logits CrossEntropyLoss를 각각 구현합니다.

### 요구사항

- probability는 `(B,C)` raw logits의 마지막 class 축 `dim=-1`에 Softmax를 적용해 만들며 모든 원소가 양수이고 각 sample 행의 합이 1이어야 합니다.
- prediction은 raw logits의 마지막 class 축에서 `argmax`를 구한 `(B,)` `torch.long` class index여야 합니다.
- CrossEntropyLoss는 probability가 아니라 `(B,C)` raw logits와 `(B,)` `torch.long` target을 직접 받아 scalar loss를 계산해야 합니다.
- logits는 `(B,C)`이고 class 수는 2 이상, target은 `(B,)` `torch.long`이며 모든 class index가 0 이상 C 미만이어야 하고 위반하면 `ValueError`로 거부해야 합니다.

### 작은 예

`(2,3)` logits에서 Softmax는 각 행의 세 class끼리 비교하므로 `dim=-1`이고 각 행의 합이 1입니다.

<details><summary>힌트 1</summary>

Softmax의 축은 batch가 아니라 class가 놓인 마지막 축입니다.

</details>

<details><summary>힌트 2</summary>

loss에는 probability 변수가 아니라 처음 받은 logits를 그대로 전달합니다.

</details>


In [ ]:
def _validate_multiclass_inputs(logits: torch.Tensor, target: torch.Tensor) -> None:
    if logits.ndim != 2 or logits.shape[1] < 2:
        raise ValueError("logits must have shape (B,C) with C >= 2")
    if target.ndim != 1 or target.shape[0] != logits.shape[0] or target.dtype != torch.long:
        raise ValueError("target must have shape (B,) and torch.long dtype")
    if bool(((target < 0) | (target >= logits.shape[1])).any()):
        raise ValueError("target class index is out of range")


def multiclass_outputs(logits: torch.Tensor, target: torch.Tensor) -> dict[str, object]:
    """Return loss, class probabilities, and predicted class indices."""
    _validate_multiclass_inputs(logits, target)

    # TODO: 마지막 class 축에 Softmax를 적용하세요.
    probability = NotImplemented

    # TODO: raw logits에서 예측 class index를 고르세요.
    prediction = NotImplemented

    # TODO: raw logits로 CrossEntropyLoss를 계산하세요.
    loss = NotImplemented
    return {"loss": loss, "probability": probability, "prediction": prediction}


In [ ]:
# sample 두 개, class 세 개
multiclass_logits = torch.tensor([[2.0, 0.5, -1.0], [0.1, 1.3, 0.2]])
multiclass_target = torch.tensor([0, 1], dtype=torch.long)
print(multiclass_outputs(multiclass_logits, multiclass_target))


In [ ]:
# 구현을 마친 뒤 이 셀을 실행하세요
def check_e11() -> None:
    logits = torch.tensor([[2.0, 0.5, -1.0], [0.1, 1.3, 0.2]])
    target = torch.tensor([0, 1], dtype=torch.long)
    result = multiclass_outputs(logits, target)
    np.testing.assert_equal(bool((result["probability"] > 0).all()), True)
    torch.testing.assert_close(result["probability"].sum(dim=-1), torch.ones(2))
    torch.testing.assert_close(result["probability"], torch.softmax(logits, dim=-1))
    torch.testing.assert_close(result["prediction"], torch.tensor([0, 1]))
    np.testing.assert_equal(result["prediction"].dtype, torch.long)
    torch.testing.assert_close(result["loss"], nn.CrossEntropyLoss()(logits, target))

    try:
        multiclass_outputs(logits, target.float())
    except ValueError:
        pass
    else:
        raise AssertionError("multiclass target이 float이면 ValueError여야 합니다")

    try:
        multiclass_outputs(logits, torch.tensor([0, 3], dtype=torch.long))
    except ValueError:
        pass
    else:
        raise AssertionError("class 범위를 벗어나면 ValueError여야 합니다")


check_e11()


### 확인 결과 정리

선택 복습입니다. 원한다면 Softmax가 필요한 목적과 `argmax(logits)`만으로 class를 고를 때 Softmax가 생략 가능한 이유를 메모해도 좋습니다. 이 메모는 실습 완료 조건이 아닙니다.


## 실습 12. 한 batch의 학습 update 수행하기

gradient 초기화부터 parameter update까지 한 번의 학습 순서를 독립적으로 연결합니다.

### 준비되어 있는 부분

target shape·dtype 검증과 함수의 최종 반환 형식만 준비되어 있습니다.

### 직접 완성할 부분

`zero_grad → forward → loss → backward → step` 전체 순서를 직접 작성하고 `loss`, `logits` 변수를 만듭니다. 검사 뒤에는 이 한 batch update가 전체 프로젝트 흐름의 어디에 놓이는지 정리합니다.

### 요구사항

- 한 batch update는 정확히 `optimizer.zero_grad() → model(inputs) → CrossEntropyLoss(raw logits, target) → loss.backward() → optimizer.step()` 순서로 한 번 실행해야 합니다.
- 함수는 update에 사용한 loss의 Python `float`와 계산 그래프에서 분리된 `(B,C)` raw logits Tensor를 반환해야 합니다.
- target이 `(B,)` `torch.long`이 아니면 update 전에 `ValueError`로 거부해야 합니다.
- 전체 학습 흐름에서는 문제와 평가 기준 정의, data·label 준비와 train/validation 분리, model·loss·optimizer 선택이 train update보다 먼저 오고, validation과 checkpoint 선택 및 새 입력 inference가 그 뒤에 온다고 설명해야 합니다.

### 작은 예

parameter는 `backward()`에서 바뀌지 않고 `.grad`가 계산된 뒤 `step()`에서 실제로 바뀝니다.

<details><summary>힌트 1</summary>

이전 batch의 `.grad`를 먼저 비운 뒤 raw logits와 loss를 계산하세요.

</details>

<details><summary>힌트 2</summary>

loss에서 backward한 다음 optimizer가 gradient를 사용하도록 마지막에 step을 호출합니다.

</details>


In [ ]:
def train_one_batch(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    inputs: torch.Tensor,
    target: torch.Tensor,
) -> tuple[float, torch.Tensor]:
    """Run exactly one multiclass parameter update."""
    if target.ndim != 1 or target.dtype != torch.long:
        raise ValueError("target must have shape (B,) and torch.long dtype")

    # TODO: 한 batch의 전체 학습 순서를 작성하세요.
    raise NotImplementedError("학습 update 순서를 구현하세요")

    return float(loss.detach()), logits.detach()


In [ ]:
# 한 번의 update 전후를 확인할 작은 모델과 batch
torch.manual_seed(12)
train_model = nn.Linear(2, 3)
train_optimizer = torch.optim.SGD(train_model.parameters(), lr=0.1)
train_inputs = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
train_target = torch.tensor([0, 2], dtype=torch.long)
print(train_one_batch(train_model, train_optimizer, train_inputs, train_target))


In [ ]:
# 구현을 마친 뒤 이 셀을 실행하세요
def check_e12() -> None:
    torch.manual_seed(12)
    model = nn.Linear(2, 3)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    inputs = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
    target = torch.tensor([0, 2], dtype=torch.long)
    before = [parameter.detach().clone() for parameter in model.parameters()]
    loss_value, logits = train_one_batch(model, optimizer, inputs, target)

    np.testing.assert_equal(any(not torch.equal(old, new) for old, new in zip(before, model.parameters())), True)
    np.testing.assert_equal(all(parameter.grad is not None for parameter in model.parameters()), True)
    np.testing.assert_equal(all(bool(torch.isfinite(parameter.grad).all()) for parameter in model.parameters()), True)
    np.testing.assert_equal(isinstance(loss_value, float), True)
    np.testing.assert_equal(tuple(logits.shape), (2, 3))
    np.testing.assert_equal(logits.requires_grad, False)

    try:
        train_one_batch(model, optimizer, inputs, target.float())
    except ValueError:
        pass
    else:
        raise AssertionError("target이 float이면 ValueError여야 합니다")


check_e12()


### 확인 결과 정리

한 batch update를 전체 학습 흐름 안에 배치해 보세요. 문제와 평가 기준을 정한 시점부터 data·label 준비, 분리, model·loss·optimizer 선택, train, validation, checkpoint, inference까지 순서와 각 단계의 목적을 적으세요.

**작성:** 아직 작성하지 않음


## 실습 13. parameter를 바꾸지 않고 validation하기

학습 때와 다른 validation의 실행 mode와 gradient 상태를 직접 제어합니다.

### 준비되어 있는 부분

target 검증, parameter 복사·불변 검사, mode 복원, accuracy 계산과 반환 형식은 준비되어 있습니다.

### 직접 완성할 부분

모델을 eval mode로 바꾸고 no-grad 문맥 안에서 forward를 실행해 `logits`를 만듭니다.

### 요구사항

- validation forward는 `model.eval()`을 적용한 뒤 `torch.no_grad()` 문맥 안에서 실행해야 하며 backward나 optimizer step을 호출하지 않아야 합니다.
- validation이 끝나면 호출 직전의 `model.training` mode를 그대로 복원해야 합니다.
- validation 전후에는 모든 model parameter 값이 같아야 합니다.
- accuracy는 class 축 argmax prediction과 target이 같은 sample의 비율인 Python `float`이고 반환 logits는 graph에서 분리된 `(B,C)` Tensor여야 합니다.
- target이 `(B,)` `torch.long`이 아니면 validation 전에 `ValueError`로 거부해야 합니다.

### 작은 예

함수를 train mode에서 호출해도 내부 forward는 eval·no-grad여야 하며 함수가 끝나면 원래 train mode로 돌아갑니다.

<details><summary>힌트 1</summary>

호출 직전 mode는 이미 저장되어 있으므로 `try` 안에서 평가 mode를 설정하세요.

</details>

<details><summary>힌트 2</summary>

forward 한 줄을 `torch.no_grad()` 문맥 안에 두면 계산 그래프를 만들지 않습니다.

</details>


In [ ]:
def validate_one_batch(
    model: nn.Module,
    inputs: torch.Tensor,
    target: torch.Tensor,
) -> tuple[float, torch.Tensor]:
    """Evaluate one multiclass batch without changing parameters."""
    if target.ndim != 1 or target.dtype != torch.long:
        raise ValueError("target must have shape (B,) and torch.long dtype")

    was_training = model.training
    before = [parameter.detach().clone() for parameter in model.parameters()]
    try:
        # TODO: eval mode와 no-grad로 validation forward를 실행하세요.
        raise NotImplementedError("validation forward를 구현하세요")
    finally:
        model.train(was_training)

    if any(not torch.equal(old, new) for old, new in zip(before, model.parameters())):
        raise RuntimeError("validation changed model parameters")
    prediction = logits.argmax(dim=-1)
    accuracy = (prediction == target).float().mean().item()
    return accuracy, logits.detach()


In [ ]:
class ValidationProbe(nn.Linear):
    def __init__(self):
        super().__init__(2, 3)
        self.forward_states: list[tuple[bool, bool]] = []

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        self.forward_states.append((self.training, torch.is_grad_enabled()))
        return super().forward(inputs)


validation_model = ValidationProbe()
validation_inputs = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
validation_target = torch.tensor([0, 2], dtype=torch.long)
print(validate_one_batch(validation_model, validation_inputs, validation_target))


In [ ]:
# 구현을 마친 뒤 이 셀을 실행하세요
def check_e13() -> None:
    torch.manual_seed(13)
    model = ValidationProbe()
    model.train()
    inputs = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
    target = torch.tensor([0, 2], dtype=torch.long)
    before = [parameter.detach().clone() for parameter in model.parameters()]
    accuracy, logits = validate_one_batch(model, inputs, target)

    np.testing.assert_equal(model.forward_states[-1], (False, False))
    np.testing.assert_equal(model.training, True)
    np.testing.assert_equal(all(torch.equal(old, new) for old, new in zip(before, model.parameters())), True)
    np.testing.assert_equal(isinstance(accuracy, float), True)
    np.testing.assert_equal(0.0 <= accuracy <= 1.0, True)
    np.testing.assert_equal(tuple(logits.shape), (2, 3))
    np.testing.assert_equal(logits.requires_grad, False)

    model.eval()
    validate_one_batch(model, inputs, target)
    np.testing.assert_equal(model.training, False)

    try:
        validate_one_batch(model, inputs, target.float())
    except ValueError:
        pass
    else:
        raise AssertionError("target이 float이면 ValueError여야 합니다")


check_e13()


### 확인 결과 정리

선택 복습입니다. 원한다면 `eval()`과 `no_grad()`가 각각 바꾸는 상태를 구분해 메모해도 좋습니다. 이 메모는 실습 완료 조건이 아닙니다.


## 실습 14. 가장 작은 validation loss checkpoint 고르기

학습 로그에서 train loss가 아니라 새 데이터 성능을 나타내는 validation loss로 저장할 epoch를 고릅니다.

### 준비되어 있는 부분

빈 history와 필수 key를 검사하고 선택된 row의 epoch를 반환하는 코드는 준비되어 있습니다.

### 직접 완성할 부분

`valid_loss`가 가장 작은 row를 선택하는 로직을 구현하고, 저장 목적에 따라 checkpoint에 어떤 state가 필요한지 구분합니다.

### 요구사항

- checkpoint는 `train_loss`가 아니라 가장 작은 `valid_loss`를 가진 history row의 `epoch`를 선택해야 합니다.
- history가 비어 있거나 row에 `epoch`, `train_loss`, `valid_loss` 중 하나가 없으면 `ValueError`로 거부해야 합니다.
- 이 실습 범위에서 추론용 checkpoint에는 최소한 model state가 필요하고 학습을 이어서 재개하려면 model state와 함께 optimizer state도 저장해야 한다고 설명해야 합니다.

### 작은 예

train loss 최소가 epoch 3이고 valid loss 최소가 epoch 2라면 checkpoint는 epoch 2입니다.

<details><summary>힌트 1</summary>

각 row에서 비교 기준으로 사용할 key는 `train_loss`가 아닙니다.

</details>

<details><summary>힌트 2</summary>

Python의 `min`에 row별 비교값을 알려주는 방법을 떠올려 보세요.

</details>


In [ ]:
def select_checkpoint_epoch(history: list[dict[str, float | int]]) -> int:
    """Return the epoch with the smallest validation loss."""
    required_keys = {"epoch", "train_loss", "valid_loss"}
    if not history or any(not required_keys.issubset(row) for row in history):
        raise ValueError("history needs epoch, train_loss, and valid_loss")

    # TODO: valid_loss가 가장 작은 row를 선택하세요.
    best_row = NotImplemented
    return int(best_row["epoch"])


In [ ]:
checkpoint_history = [
    {"epoch": 1, "train_loss": 0.8, "valid_loss": 0.7},
    {"epoch": 2, "train_loss": 0.5, "valid_loss": 0.4},
    {"epoch": 3, "train_loss": 0.3, "valid_loss": 0.6},
]
print("checkpoint epoch:", select_checkpoint_epoch(checkpoint_history))


In [ ]:
# 구현을 마친 뒤 이 셀을 실행하세요
def check_e14() -> None:
    history = [
        {"epoch": 1, "train_loss": 0.8, "valid_loss": 0.7},
        {"epoch": 2, "train_loss": 0.5, "valid_loss": 0.4},
        {"epoch": 3, "train_loss": 0.3, "valid_loss": 0.6},
    ]
    np.testing.assert_equal(select_checkpoint_epoch(history), 2)
    np.testing.assert_equal(
        select_checkpoint_epoch([{"epoch": 7, "train_loss": 0.2, "valid_loss": 0.3}]),
        7,
    )

    try:
        select_checkpoint_epoch([])
    except ValueError:
        pass
    else:
        raise AssertionError("빈 history는 ValueError여야 합니다")

    try:
        select_checkpoint_epoch([{"epoch": 1, "train_loss": 0.2}])
    except ValueError:
        pass
    else:
        raise AssertionError("valid_loss가 없으면 ValueError여야 합니다")


check_e14()


### 확인 결과 정리

checkpoint의 저장 목적에 따라 필요한 state를 구분해 보세요. 학습이 끝난 모델로 inference만 할 때와, 같은 optimizer 상태에서 학습을 이어서 재개할 때 각각 무엇을 저장해야 하나요?

**작성:** 아직 작성하지 않음
